<a href="https://colab.research.google.com/github/kerenG99/UFZ-Helmoltz-Summer-School-2026/blob/main/notebooks/04_visualization_ggplot2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04 · Visualization with ggplot2

> **Trends in multi-omics data analysis for Microbial Ecology and Biotechnology** - Summer School, UFZ Leipzig  
> *Refreshing / Basic Knowledge: R* - Instructor: Anderson Santos

## Setup — run this first

This cell installs the R packages the lessons use and makes the example data available. On **Google Colab** it just works: run it (Shift+Enter) and wait until it prints **Ready**. You do not need to understand it.

In [ ]:
# Setup — run this first. Works on Google Colab and on a local Jupyter with R.
# It installs the packages the lessons use and makes the example data available.

pkgs <- c("readr", "dplyr", "tidyr", "tibble", "ggplot2", "forcats", "stringr", "vegan")
to_install <- setdiff(pkgs, rownames(installed.packages()))
if (length(to_install) > 0) {
  # On Colab (Linux) use Posit Public Package Manager binaries: ~1-2 min instead of
  # compiling from source (~15-20 min).
  if (grepl("linux", R.version$os) && file.exists("/etc/os-release")) {
    cn <- grep("^VERSION_CODENAME=", readLines("/etc/os-release"), value = TRUE)
    cn <- gsub('VERSION_CODENAME=|"', "", cn)
    if (length(cn) == 1 && nzchar(cn)) {
      options(repos = c(CRAN = sprintf("https://p3m.dev/cran/__linux__/%s/latest", cn)))
      options(HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(),
              paste(getRversion(), R.version$platform, R.version$arch, R.version$os)))
    }
  }
  install.packages(to_install)
}

# Fetch the example data if it is not already next to the notebook (the Colab case).
if (!file.exists("../data/sample_metadata.csv") && !file.exists("data/sample_metadata.csv")) {
  system("git clone -q https://github.com/andersonavilasantos/ufz-summerschool-r.git course_repo")
  setwd("course_repo/notebooks")
}

# Load the packages quietly and keep read_csv output tidy (cleaner notebook output).
suppressPackageStartupMessages(invisible(lapply(pkgs, require, character.only = TRUE)))
options(readr.show_col_types = FALSE)

cat("Ready. Data folder:", if (file.exists("../data/sample_metadata.csv")) "../data" else "data", "\n")

In [ ]:
#changes to upload to code

## Learning objectives

- Understand the **grammar of graphics**: data + aesthetics + geoms + facets + scales + theme.
- Build the everyday plots: **scatter**, **histogram**, **boxplot**, **bar** (and stacked bar).
- Map variables to **colour / fill / shape** and split panels with **faceting**.
- Control **scales**, **labels**, **themes**, and **save** a figure to a file.

---

## 0 · Setup — data ready to plot

We reload the data and rebuild the two tidy tables we made in lesson 03 (the
per-sample metadata and the long, joined abundance table), so this lesson stands
on its own.

In [ ]:
library(readr); library(dplyr); library(tidyr)   # import + wrangle (from lessons 02-03)
library(ggplot2); library(forcats)                # ggplot2 = plotting; forcats = order factors

# Locate and load the three tables, exactly as before.
data_dir <- if (file.exists("../data/sample_metadata.csv")) "../data" else "data"
meta     <- read_csv(file.path(data_dir, "sample_metadata.csv"))
taxonomy <- read_csv(file.path(data_dir, "taxonomy.csv"))
abund    <- read_csv(file.path(data_dir, "genus_abundance.csv"))

# Re-apply the clinical BMI order so plots list groups low -> high, not alphabetically.
bmi_levels <- c("underweight","lean","overweight","obese","severeobese","morbidobese")
meta <- meta |> mutate(bmi_group = factor(bmi_group, levels = bmi_levels))

# Rebuild the long, joined table from lesson 03 (pivot -> join taxonomy -> join metadata),
# so this lesson is self-contained: one row per genus-in-sample, with phylum + region.
full <- abund |>
  pivot_longer(-sample_id, names_to = "genus", values_to = "abundance") |>
  left_join(taxonomy, by = "genus") |>
  left_join(select(meta, sample_id, nationality, bmi_group, age, diversity_shannon),
            by = "sample_id")

---

## 1 · The grammar of graphics in one picture

Every ggplot is built from layers you **add with `+`**:

1. `ggplot(data, aes(...))` — the **data** and the **aesthetic mapping**
   (which variable goes on x, y, colour, …).
2. a **geom** — `geom_point()`, `geom_col()`, `geom_boxplot()`, … — the shapes.
3. optional **facets**, **scales**, **labels** and a **theme**.

In [ ]:
# ggplot(data, aes(...)) sets the data + the aesthetic mapping (which var -> which axis).
# Then '+' adds a layer. Note: ggplot chains with '+', not the pipe |>.
ggplot(meta, aes(x = age, y = diversity_shannon)) +  # map age -> x, diversity -> y
  geom_point()                                        # draw one point per sample

Read it as a sentence: *"For each sample, put `age` on x and `diversity_shannon`
on y, and draw a point."* Everything else is refinement.

> **Instructor note.** The `+` (not the pipe) chains ggplot layers. A common
> beginner slip is writing `|>` between layers, so flag it early.

---

## 2 · Scatter plots — two numbers, plus a category by colour

Map a third variable to **colour** and add a trend line with `geom_smooth()`.

In [ ]:
# You can pipe data into ggplot(), then switch to '+' once the plot starts.
meta |>
  filter(!is.na(age)) |>                                # drop samples with unknown age first
  ggplot(aes(x = age, y = diversity_shannon)) +
  geom_point(alpha = 0.4, colour = "steelblue") +      # alpha < 1 = semi-transparent points
  geom_smooth(method = "lm", se = TRUE, colour = "darkred") +  # lm = straight-line fit + CI band
  labs(title = "Gut diversity vs. age",                # labs() sets titles and axis labels
       x = "Age (years)", y = "Shannon diversity")

Colour by a category to compare groups in one panel:

In [ ]:
meta |>
  filter(!is.na(age), nationality %in% c("Scandinavia","US","SouthEurope")) |>  # 3 regions
  # Mapping colour = nationality inside aes() colours points by category automatically.
  ggplot(aes(x = age, y = diversity_shannon, colour = nationality)) +
  geom_point(alpha = 0.6) +
  labs(title = "Diversity vs. age by region", colour = "Region")  # 'colour =' renames the legend

### Exercise 1

Make a scatter plot of `age` (x) vs `diversity_shannon` (y), coloured by
`bmi_group`. Drop rows where `age` or `bmi_group` is missing.

<details><summary><b>Solution</b> (click to expand)</summary>


```r
meta |>
  filter(!is.na(age), !is.na(bmi_group)) |>
  ggplot(aes(age, diversity_shannon, colour = bmi_group)) +
  geom_point(alpha = 0.6) +
  labs(colour = "BMI group")
```
</details>

---

## 3 · Distributions — histogram and density

One numeric variable: how is it spread?

In [ ]:
# A histogram needs only one variable (x); ggplot bins it and counts per bin.
ggplot(meta, aes(x = diversity_shannon)) +
  geom_histogram(bins = 30, fill = "steelblue", colour = "white") +  # 30 bars; white bar borders
  labs(title = "Distribution of Shannon diversity",
       x = "Shannon diversity", y = "Number of samples")

Overlay densities to compare groups (fill with transparency):

In [ ]:
meta |>
  filter(nationality %in% c("Scandinavia","US")) |>
  # fill = nationality draws one smoothed density curve per region, in different colours.
  ggplot(aes(x = diversity_shannon, fill = nationality)) +
  geom_density(alpha = 0.4) +                          # alpha lets overlapping curves show through
  labs(title = "Diversity distribution: Scandinavia vs US", fill = "Region")

---

## 4 · Boxplots — a number across categories

Boxplots are the workhorse for "does this metric differ between groups?". Add the
raw points with `geom_jitter()` for honesty about sample sizes.

In [ ]:
meta |>
  filter(!is.na(bmi_group)) |>
  # x = a category, y = a number: the classic "does this metric differ between groups?" plot.
  ggplot(aes(x = bmi_group, y = diversity_shannon, fill = bmi_group)) +
  geom_boxplot(outlier.alpha = 0.3) +                  # box = median + quartiles; faded outliers
  geom_jitter(width = 0.15, alpha = 0.15, size = 0.5) +# scatter the raw points so N is visible
  labs(title = "Diversity across BMI groups",
       x = "BMI group", y = "Shannon diversity") +
  theme(legend.position = "none")            # x-axis already labels the groups -> hide the legend

Reorder categories by their median with `forcats::fct_reorder()` so the plot
tells the story at a glance:

In [ ]:
meta |>
  # fct_reorder(f, x, .fun) reorders the category f by a summary of x — here each
  # region is ordered by its median diversity, so the boxes climb in a readable order.
  ggplot(aes(x = fct_reorder(nationality, diversity_shannon, .fun = median),
             y = diversity_shannon)) +
  geom_boxplot(fill = "seagreen", alpha = 0.7) +
  coord_flip() +                              # flip x/y -> horizontal bars, long labels fit
  labs(title = "Gut diversity by region (sorted by median)",
       x = NULL, y = "Shannon diversity")     # x = NULL: no axis title needed (labels are clear)

### Exercise 2

Draw a boxplot of `diversity_shannon` by `sex` (drop missing sex). Fill by sex
and hide the legend.

<details><summary><b>Solution</b> (click to expand)</summary>


```r
meta |>
  filter(!is.na(sex)) |>
  ggplot(aes(sex, diversity_shannon, fill = sex)) +
  geom_boxplot() +
  theme(legend.position = "none")
```
</details>

---

## 5 · Bar charts — counting and summarising categories

`geom_bar()` **counts** rows per category; `geom_col()` draws bars from values you
**already computed**. Use `geom_col()` after a `summarise()`.

In [ ]:
# geom_bar counts rows for you: give it only x, no y needed.
meta |>
  ggplot(aes(x = fct_infreq(nationality))) +   # fct_infreq: order bars from most to least frequent
  geom_bar(fill = "darkorange") +              # bar height = number of samples in that region
  labs(title = "Samples per region", x = NULL, y = "Count") +
  theme(axis.text.x = element_text(angle = 30, hjust = 1))  # tilt x labels 30 deg so they don't overlap

In [ ]:
# geom_col draws bars from values you already computed -> summarise first, then plot.
meta |>
  group_by(nationality) |>
  summarise(mean_shannon = mean(diversity_shannon)) |>  # one mean per region
  ggplot(aes(x = fct_reorder(nationality, mean_shannon), y = mean_shannon)) +  # order bars by height
  geom_col(fill = "steelblue") +               # bar height = the mean we computed (y)
  coord_flip() +                               # horizontal for readable region names
  labs(title = "Mean Shannon diversity per region", x = NULL, y = "Mean diversity")

### 5.1 · Stacked bars — taxonomic composition

The classic microbiome figure: **phylum composition per region** (the table we
built in lesson 03). We compute relative abundances, then stack them.

In [ ]:
# Same split-apply-combine as lesson 03: per-phylum sum -> per-sample proportion -> region mean.
composition <- full |>
  group_by(sample_id, nationality, phylum) |>
  summarise(phy_abund = sum(abundance), .groups = "drop") |>  # total abundance per phylum in a sample
  group_by(sample_id) |>
  mutate(rel = phy_abund / sum(phy_abund)) |>                 # -> its share of that sample
  group_by(nationality, phylum) |>
  summarise(mean_rel = mean(rel), .groups = "drop")           # -> average share per region

# fill = phylum stacks phyla within each region bar; fct_reorder orders the stack by size.
ggplot(composition, aes(x = nationality, y = mean_rel,
                        fill = fct_reorder(phylum, mean_rel, sum))) +
  geom_col() +                                   # bars stack automatically when 'fill' varies
  scale_y_continuous(labels = scales::percent) + # format the y axis as percentages
  labs(title = "Phylum composition across regions",
       x = NULL, y = "Mean relative abundance", fill = "Phylum") +
  theme(axis.text.x = element_text(angle = 30, hjust = 1))

> **Instructor note.** This figure ties the session together: the wrangling from
> lesson 03 feeds directly into a publication-style plot.

---

## 6 · Faceting — small multiples

`facet_wrap()` splits one plot into a panel per category — often clearer than
cramming everything into one busy chart.

In [ ]:
meta |>
  filter(!is.na(age), nationality %in% c("Scandinavia","US","SouthEurope","UKIE")) |>
  ggplot(aes(age, diversity_shannon)) +
  geom_point(alpha = 0.4, colour = "steelblue") +
  geom_smooth(method = "lm", se = FALSE, colour = "darkred") +  # se = FALSE: line only, no band
  facet_wrap(~ nationality) +                    # ~var splits the plot into one panel per region
  labs(title = "Diversity vs age, by region")

---

## 7 · Scales, labels and themes — polish

- **Scales** control how data maps to colour/position (`scale_fill_brewer`,
  `scale_x_log10`, …).
- **`labs()`** sets titles and axis/legend labels.
- **Themes** (`theme_minimal`, `theme_bw`, …) set the overall look.

In [ ]:
# Store the plot in an object 'p' so we can print it and save it later (§8).
p <- composition |>
  filter(phylum %in% c("Firmicutes", "Bacteroidetes", "Proteobacteria",
                       "Actinobacteria")) |>                # focus on the four major phyla
  ggplot(aes(x = nationality, y = mean_rel, fill = phylum)) +
  geom_col(position = "dodge") +                          # "dodge" = side-by-side, not stacked
  scale_y_continuous(labels = scales::percent) +          # y axis as %
  scale_fill_brewer(palette = "Set2") +                   # a colour-blind-friendly fill palette
  labs(title = "Four major phyla across regions",
       subtitle = "Human gut microbiome — HITChip Atlas",
       x = NULL, y = "Mean relative abundance", fill = "Phylum",
       caption = "Data: Lahti et al. 2014, Nat. Commun.") +
  theme_minimal(base_size = 12) +                         # a clean theme; base_size scales all text
  theme(axis.text.x = element_text(angle = 30, hjust = 1))

p                                                         # printing the object draws the plot

---

## 8 · Saving a figure

`ggsave()` writes the last (or a named) plot to a file — PNG for slides, PDF for
print. Size and resolution are up to you.

In [ ]:
dir.create("../figures", showWarnings = FALSE)   # make the folder; stay quiet if it already exists
# Choose an output path that works whether or not ../figures exists.
out <- ifelse(dir.exists("../figures"),
              "../figures/phylum_composition.png", "phylum_composition.png")
ggsave(out, plot = p, width = 8, height = 5, dpi = 150)  # write plot 'p' to disk (inches, dpi)
cat("Saved figure to", normalizePath(out), "\n")         # print the full path we wrote to

### Exercise 3

Recreate the mean-diversity-per-region bar chart from §5, but (a) use
`theme_bw()`, (b) colour the bars by region, and (c) add a proper title and axis
labels. Then save it to `../figures/diversity_by_region.png`.

<details><summary><b>Solution</b> (click to expand)</summary>


```r
g <- meta |>
  group_by(nationality) |>
  summarise(mean_shannon = mean(diversity_shannon)) |>
  ggplot(aes(fct_reorder(nationality, mean_shannon), mean_shannon,
             fill = nationality)) +
  geom_col() + coord_flip() +
  labs(title = "Mean gut diversity per region",
       x = NULL, y = "Mean Shannon diversity") +
  theme_bw() + theme(legend.position = "none")
ggsave(ifelse(dir.exists("../figures"), "../figures/diversity_by_region.png",
              "diversity_by_region.png"), g, width = 7, height = 4)
g
```
</details>

---

## Recap

- ggplot = **data + `aes()` + geoms** (`+` to add layers).
- **Scatter** for two numbers, **histogram/density** for one, **boxplot** for a
  number across categories, **bar/col** for counts and summaries,
  **stacked bar** for composition.
- Map categories to **colour/fill**; split panels with **`facet_wrap`**.
- **`fct_reorder`** sorts categories so the plot reads itself.
- Polish with **scales**, **`labs()`**, **themes**; export with **`ggsave()`**.

**Next:** lesson **05** — the guided **capstone**: import → clean → wrangle →
summarise → figures → a simple stat/ordination → export, on the real data.